# Kaggle MiniMax H3 on Kaggle

This notebook is intentionally conservative about the two T4 architecture. It evaluates SGLang, Diffusers balanced placement, automatic Diffusers H3 transformer-block dispatch, Diffusers explicit component placement, and the optional ComfyUI sequence-parallel node before allowing a generation. Native ComfyUI one-GPU CPU offload is an explicit fallback only.

Set `GITHUB_REPOSITORY` in the first code cell to a public GitHub repository containing this package to clone it directly into writable `/kaggle/working`; no Kaggle Dataset upload is then required. If it is left empty, the cell uses the uploaded `kaggle_h3` dataset as a fallback.

The notebook writes all artifacts to `kaggle_h3/results/`.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
os.environ.setdefault('KAGGLE_H3_MODEL_REVISION', 'a98869194787969724c7425d95d0ed73ce9202af')

PACKAGE_NAME = 'kaggle_h3'
# Preferred source: a public GitHub repository containing this package.
# Example: https://github.com/your-account/kaggle_h3.git
GITHUB_REPOSITORY = os.environ.get('KAGGLE_H3_GITHUB_REPOSITORY', 'https://github.com/celunah/kaggle_h3.git').strip()
GITHUB_REF = os.environ.get('KAGGLE_H3_GITHUB_REF', 'main').strip() or 'main'
GITHUB_ROOT = Path('/kaggle/working') / f'{PACKAGE_NAME}_github'
WORKING_ROOT = Path('/kaggle/working') / PACKAGE_NAME
INPUT_ROOT = Path('/kaggle/input') / PACKAGE_NAME

def _is_package_root(path):
    return (path / 'src').is_dir() and (path / 'requirements-kaggle.txt').is_file()

def _ensure_github_source(repository, destination, revision):
    destination = destination.resolve()
    git_dir = destination / '.git'
    is_commit = len(revision) == 40 and all(c in '0123456789abcdefABCDEF' for c in revision)
    if not git_dir.is_dir():
        if destination.exists() and any(destination.iterdir()):
            raise RuntimeError(f'GitHub destination is non-empty and is not a Git checkout: {destination}')
        destination.parent.mkdir(parents=True, exist_ok=True)
        command = ['git', 'clone', '--depth', '1']
        if not is_commit:
            command += ['--branch', revision]
        command += [repository, str(destination)]
        subprocess.run(command, check=True)
        if is_commit:
            subprocess.run(['git', '-C', str(destination), 'fetch', '--depth', '1', 'origin', revision], check=True)
            subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
        status = 'cloned'
    else:
        remote = subprocess.run(['git', '-C', str(destination), 'config', '--get', 'remote.origin.url'], check=True, capture_output=True, text=True).stdout.strip()
        normalize = lambda value: value.strip().removesuffix('.git').rstrip('/')
        if remote and normalize(remote) != normalize(repository):
            raise RuntimeError(f'Existing GitHub checkout points to {remote}, not {repository}.')
        subprocess.run(['git', '-C', str(destination), 'fetch', '--depth', '1', 'origin', revision], check=True)
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
        status = 'updated'
    resolved = subprocess.run(['git', '-C', str(destination), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
    print(f'GitHub package {status}: {repository}@{resolved}')
    return destination

# Tolerate a different dataset slug and any nested-folder upload layout.
# We locate the package by its required-file marker instead of assuming
# that Kaggle mounted the uploaded folder at a particular depth.
# If an older dataset version left a second valid root mounted, prefer the
# explicitly renamed kaggle_h3 root; /kaggle/input is read-only.
if not _is_package_root(INPUT_ROOT) and Path('/kaggle/input').is_dir():
    matches = []
    for marker in Path('/kaggle/input').rglob('requirements-kaggle.txt'):
        root = marker.parent
        if _is_package_root(root) and root not in matches:
            matches.append(root)
    if len(matches) == 1:
        INPUT_ROOT = matches[0]
    elif len(matches) > 1:
        current_roots = [root for root in matches if root.name.lower() == PACKAGE_NAME.lower()]
        if len(current_roots) == 1:
            INPUT_ROOT = current_roots[0]
            ignored_roots = [root for root in matches if root != INPUT_ROOT]
            print('Selected current package root:', INPUT_ROOT)
            print('Ignoring older/duplicate package roots:', ignored_roots)
        else:
            raise RuntimeError(f'Multiple current kaggle_h3 package roots found under /kaggle/input: {matches}')

# Kaggle Dataset mounts are read-only. Refresh the writable working copy
# from the current dataset version so corrected node files are not stale.
# dirs_exist_ok preserves generated results that are not in the dataset.
if _is_package_root(INPUT_ROOT):
    WORKING_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(INPUT_ROOT, WORKING_ROOT, dirs_exist_ok=True)
    print('Copied/refreshed package:', INPUT_ROOT, '->', WORKING_ROOT)

if GITHUB_REPOSITORY:
    PROJECT_ROOT = _ensure_github_source(GITHUB_REPOSITORY, GITHUB_ROOT, GITHUB_REF)
    if not _is_package_root(PROJECT_ROOT):
        raise RuntimeError('The GitHub checkout does not contain src/ and requirements-kaggle.txt at its root.')
    print('Using GitHub source; Kaggle Dataset input is not required.')
else:
    candidates = [
        WORKING_ROOT,
        Path.cwd(),
    ]
    PROJECT_ROOT = next((p.resolve() for p in candidates if (p / 'src').is_dir()), None)
    if PROJECT_ROOT is None:
        for parent in [Path.cwd(), *Path.cwd().parents]:
            candidate = parent / 'kaggle_h3'
            if (candidate / 'src').is_dir():
                PROJECT_ROOT = candidate.resolve()
                break
if PROJECT_ROOT is None:
    input_listing = []
    if Path('/kaggle/input').is_dir():
        input_listing = [str(path) for path in Path('/kaggle/input').iterdir()]
    raise FileNotFoundError(
        'Could not find a package containing src/ and requirements-kaggle.txt. '
        f'Kaggle input mounts: {input_listing}'
    )
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULTS_DIR =', PROJECT_ROOT / 'results')

In [ ]:
# The package dependencies are small. The optional layer-sharded backend is
# installed separately so ComfyUI-only sessions do not silently replace
# their Torch/Diffusers stack.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-kaggle.txt')], check=True)
INSTALL_LAYER_SHARDED_BACKEND = True
if INSTALL_LAYER_SHARDED_BACKEND:
    optional_requirements = PROJECT_ROOT / 'requirements-h3-layer-sharded.txt'
    optional_install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(optional_requirements)], check=False, capture_output=True, text=True)
    if optional_install.returncode == 0:
        print('Automatic layer-sharded Diffusers backend dependencies installed.')
    else:
        print('Automatic layer-sharded backend unavailable; optional install failed:')
        print((optional_install.stderr or optional_install.stdout)[-2000:])
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('HF_TOKEN loaded from Kaggle Secrets without printing its value.')
except Exception:
    print('No HF_TOKEN secret loaded; public model files remain usable.')

from kaggle_h3.backends import evaluate_backends
print('Direct ComfyUI startup mode: enabled; no generation request will be queued.')

## Hardware and backend preflight

This preflight prints CUDA devices and planned maps for every backend before direct ComfyUI startup. Observed same-generation use remains false until a real worker generation records both GPUs. A path is not called eligible just because two GPUs are visible.

In [ ]:
from kaggle_h3.bootstrap import detect_hardware
HARDWARE = detect_hardware()
EVALUATION = evaluate_backends(HARDWARE, comfy_root=PROJECT_ROOT / 'ComfyUI')
print(json.dumps(HARDWARE, indent=2, default=str))
for item in EVALUATION['attempts']:
    print('\nBACKEND:', item['backend'])
    print('status:', item.get('status'), '| declared capability:', item.get('same_generation_multi_gpu'), '| observed:', item.get('observed_same_generation_multi_gpu'), '| generation attempted:', item.get('generation_attempted'))
    print('detected devices:', json.dumps(item.get('detected_cuda_devices', []), default=str))
    print('planned device map:', json.dumps(item.get('actual_device_map'), indent=2, default=str))
    print('observed device map:', json.dumps(item.get('observed_device_map'), indent=2, default=str))
    print('capacity:', item.get('capacity_eligible'), '| reason:', item.get('reason'))

## Start ComfyUI

This cell installs the pinned ComfyUI checkout and H3 adapter node, downloads only the active Ref2VA or FL2VA model file into Kaggle scratch storage, and starts the ComfyUI web server. It does not create or queue any request.

The model tree lives under /kaggle/tmp, not /kaggle/working. Kaggle scratch storage is available for this session but is not persistent between sessions.

Native ComfyUI is started with low-VRAM and CPU-VAE offload as an explicit one-GPU fallback. The hardware preflight above remains the source of truth for multi-GPU capability.

In [ ]:
import importlib
import kaggle_h3.bootstrap as _kaggle_bootstrap
_kaggle_bootstrap = importlib.reload(_kaggle_bootstrap)
from kaggle_h3.bootstrap import (
    configure_comfyui_model_paths,
    download_selected_models,
    ensure_comfyui,
    install_comfyui_dependencies,
    install_h3_adapter_node,
    MODE_MODEL_FILES,
    plan_execution_devices,
    start_comfyui,
    wait_for_comfyui_device_visibility,
)

COMFY_ROOT = PROJECT_ROOT / 'ComfyUI'
MODEL_STORE_ROOT = Path('/kaggle/tmp/minimax-h3-models')
MODEL_ROOT = MODEL_STORE_ROOT / 'models'
if 'COMFY_PROCESS' in globals() and COMFY_PROCESS.process.poll() is None:
    print('Stopping existing ComfyUI before changing the active H3 checkpoint.')
    COMFY_PROCESS.stop()
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
# Keep the shared Qwen/video/audio files, but store only one large
# diffusion partition at a time. FL2VA also serves T2VA. Change this
# value to Ref2VA when that workflow is needed, then rerun this cell.
ACTIVE_H3_MODE = 'Ref2VA'
if ACTIVE_H3_MODE not in ('FL2VA', 'Ref2VA'):
    raise ValueError('ACTIVE_H3_MODE must be FL2VA or Ref2VA')
PRUNE_INACTIVE_DIFFUSION_MODEL = True
if PRUNE_INACTIVE_DIFFUSION_MODEL:
    for mode, (directory, filename) in MODE_MODEL_FILES.items():
        if mode == ACTIVE_H3_MODE:
            continue
        inactive_path = MODEL_ROOT / directory / filename
        if inactive_path.is_file():
            print('Removing inactive diffusion checkpoint:', inactive_path)
            inactive_path.unlink()
MODEL_MODES = (ACTIVE_H3_MODE,)

COMFY_SETUP = ensure_comfyui(COMFY_ROOT)
MODEL_STATUS = {
    mode: download_selected_models(COMFY_ROOT, mode, models_root=MODEL_ROOT)
    for mode in MODEL_MODES
}
MODEL_PATHS_STATUS = configure_comfyui_model_paths(COMFY_ROOT, MODEL_ROOT)
NODE_STATUS = install_h3_adapter_node(COMFY_ROOT, PROJECT_ROOT)
DEPENDENCY_STATUS = install_comfyui_dependencies(COMFY_ROOT)
# The direct ComfyUI frontend exposes both T4s so the custom H3 phase sampler
# can dispatch intact transformer blocks and verify both devices at runtime.
EXECUTION_PLAN = plan_execution_devices(HARDWARE, backend_supports_sharding=True)
detected_device_ids = [int(device['index']) for device in HARDWARE.get('cuda_devices', [])]
planned_device_ids = [int(device_id) for device_id in EXECUTION_PLAN.get('visible_device_ids', [])]
print('Preflight CUDA device ids:', detected_device_ids)
print('ComfyUI planned CUDA device ids:', planned_device_ids)
if len(detected_device_ids) >= 2 and len(planned_device_ids) < 2:
    raise RuntimeError(
        'The notebook detected two GPUs, but the execution plan contains fewer than two. '
        'Refresh the kaggle_h3 dataset copy before starting ComfyUI.'
    )
os.environ['KAGGLE_H3_PHASE_SHARDING'] = 'auto'
os.environ['KAGGLE_H3_PHASE_OFFLOAD_DIR'] = '/kaggle/tmp/kaggle_h3-layer-offload/comfyui'

COMFY_PROCESS = start_comfyui(
    COMFY_ROOT,
    port=8188,
    listen_host='0.0.0.0',
    execution_plan=EXECUTION_PLAN,
    cpu_vae=False,
    log_dir=PROJECT_ROOT / 'results',
)
if len(planned_device_ids) >= 2:
    COMFY_DEVICE_STATUS = wait_for_comfyui_device_visibility(
        COMFY_PROCESS, expected_count=len(planned_device_ids)
    )
    print('ComfyUI process-visible devices:', json.dumps(COMFY_DEVICE_STATUS['devices'], indent=2, default=str))

print('ComfyUI started without queuing a request.')
print('ComfyUI is listening inside the Kaggle runtime at http://127.0.0.1:8188.')
print('Run the optional public-tunnel cell below if you want to test remote access.')
print('ComfyUI log:', COMFY_PROCESS.log_path)
print(json.dumps({
    'comfyui': COMFY_SETUP,
    'active_h3_mode': ACTIVE_H3_MODE,
    'models': MODEL_STATUS,
    'model_paths': MODEL_PATHS_STATUS,
    'h3_adapter_node': NODE_STATUS,
    'dependencies': DEPENDENCY_STATUS,
    'execution_plan': EXECUTION_PLAN,
}, indent=2, default=str))

## Optional public tunnel

Run this cell only after the ComfyUI startup cell. It attempts to bind a temporary Cloudflare Quick Tunnel to ComfyUI's internal port 8188 and prints the public URL. Kaggle or its network policy may reject the tunnel; a failure is reported instead of being hidden. Anyone who receives the URL can reach the exposed ComfyUI service, so stop the tunnel when finished.

In [ ]:
import re, shutil, stat, subprocess, time, urllib.request

if 'COMFY_PROCESS' not in globals() or COMFY_PROCESS.process.poll() is not None:
    raise RuntimeError('Run the ComfyUI startup cell first; port 8188 is not running.')

# Re-running this cell replaces the prior tunnel instead of creating duplicates.
old_tunnel = globals().get('CLOUDFLARED_PROCESS')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()

# ComfyUI's default origin/CSRF guard rejects the dynamic tunnel hostname.
# Restart only for this explicit public-tunnel attempt; normal startup stays protected.
if not globals().get('COMFY_PUBLIC_TUNNEL_MODE', False):
    COMFY_PROCESS.stop()
    import inspect
    if 'enable_cors_header' in inspect.signature(start_comfyui).parameters:
        COMFY_PROCESS = start_comfyui(
            COMFY_ROOT,
            port=8188,
            listen_host='0.0.0.0',
            execution_plan=EXECUTION_PLAN,
            log_dir=PROJECT_ROOT / 'results',
            enable_cors_header='*',
        )
    else:
        # Compatibility path for a Kaggle package copy from before the CORS parameter.
        from kaggle_h3.bootstrap import ComfyProcess, comfy_launch_command
        fallback_command = comfy_launch_command(
            COMFY_ROOT,
            8188,
            listen_host='0.0.0.0',
            visible_device_ids=EXECUTION_PLAN.get('visible_device_ids', []),
        )
        fallback_command.extend(['--enable-cors-header', '*'])
        launch_env = os.environ.copy()
        visible_device_ids = EXECUTION_PLAN.get('visible_device_ids', [])
        if visible_device_ids:
            launch_env['CUDA_VISIBLE_DEVICES'] = ','.join(str(item) for item in visible_device_ids)
        launch_env['PYTHONUNBUFFERED'] = '1'
        fallback_log_path = (PROJECT_ROOT / 'results' / 'comfyui.log').resolve()
        with fallback_log_path.open('a', encoding='utf-8') as fallback_log:
            fallback_process = subprocess.Popen(
                fallback_command,
                cwd=COMFY_ROOT,
                env=launch_env,
                stdout=fallback_log,
                stderr=subprocess.STDOUT,
                text=True,
            )
        time.sleep(1.0)
        if fallback_process.poll() is not None:
            raise RuntimeError(f'ComfyUI exited with code {fallback_process.returncode}; see {fallback_log_path}')
        COMFY_PROCESS = ComfyProcess(process=fallback_process, log_path=fallback_log_path)
        print('Using compatibility launch path for the older Kaggle package copy.')
    COMFY_PUBLIC_TUNNEL_MODE = True
    print('Restarted ComfyUI with --enable-cors-header * for the public tunnel.')
    print('WARNING: anyone with the tunnel URL can access this unauthenticated ComfyUI instance.')

COMFY_LOCAL_URL = 'http://127.0.0.1:8188'
print('Waiting for ComfyUI local health at', COMFY_LOCAL_URL)
comfy_ready = False
comfy_deadline = time.time() + 120
while time.time() < comfy_deadline:
    if COMFY_PROCESS.process.poll() is not None:
        print('ComfyUI exited with code:', COMFY_PROCESS.process.returncode)
        print('ComfyUI log:', COMFY_PROCESS.log_path)
        print(COMFY_PROCESS.log_path.read_text(encoding='utf-8', errors='replace')[-6000:])
        raise RuntimeError('ComfyUI is not running; do not start the tunnel.')
    try:
        with urllib.request.urlopen(COMFY_LOCAL_URL + '/system_stats', timeout=3) as response:
            comfy_ready = 200 <= response.status < 300
    except Exception:
        comfy_ready = False
    if comfy_ready:
        break
    time.sleep(2)
if not comfy_ready:
    print('ComfyUI did not answer /system_stats within 120 seconds.')
    print('ComfyUI log:', COMFY_PROCESS.log_path)
    print(COMFY_PROCESS.log_path.read_text(encoding='utf-8', errors='replace')[-6000:])
    raise RuntimeError('Local ComfyUI is not ready; tunnel was not started.')
print('Local ComfyUI health check passed.')
if len(globals().get('planned_device_ids', [])) >= 2:
    COMFY_DEVICE_STATUS = wait_for_comfyui_device_visibility(
        COMFY_PROCESS, expected_count=len(planned_device_ids)
    )
    print('Tunnel restart preserved ComfyUI devices:', json.dumps(COMFY_DEVICE_STATUS['devices'], indent=2, default=str))

CLOUDFLARED_PATH = Path('/kaggle/tmp/cloudflared')
TUNNEL_LOG = PROJECT_ROOT / 'results' / 'cloudflared.log'
TUNNEL_LOG.parent.mkdir(parents=True, exist_ok=True)
CLOUDFLARED_COMMAND = shutil.which('cloudflared')
if CLOUDFLARED_COMMAND is None:
    if not CLOUDFLARED_PATH.exists():
        print('Downloading cloudflared into /kaggle/tmp ...')
        urllib.request.urlretrieve(
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            CLOUDFLARED_PATH,
        )
        CLOUDFLARED_PATH.chmod(CLOUDFLARED_PATH.stat().st_mode | stat.S_IXUSR)
    CLOUDFLARED_COMMAND = str(CLOUDFLARED_PATH)

with TUNNEL_LOG.open('w', encoding='utf-8') as tunnel_log:
    CLOUDFLARED_PROCESS = subprocess.Popen(
        [
            CLOUDFLARED_COMMAND,
            'tunnel',
            '--url',
            COMFY_LOCAL_URL,
            '--no-autoupdate',
        ],
        stdout=tunnel_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

public_url = None
deadline = time.time() + 30
while time.time() < deadline and CLOUDFLARED_PROCESS.poll() is None:
    log_text = TUNNEL_LOG.read_text(encoding='utf-8', errors='replace')
    match = re.search(r'https://[A-Za-z0-9.-]+\.trycloudflare\.com', log_text)
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)

if public_url:
    print('ComfyUI public URL:', public_url)
    print('Tunnel PID:', CLOUDFLARED_PROCESS.pid)
else:
    status = CLOUDFLARED_PROCESS.poll()
    print('Cloudflare tunnel did not provide a public URL within 30 seconds.')
    print('Process status:', status if status is not None else 'still running')
    print('Tunnel log:', TUNNEL_LOG)
    print(TUNNEL_LOG.read_text(encoding='utf-8', errors='replace')[-4000:])
